# Hypersimplex family — full exploration, Δ(2,n) for n = 4, ..., 9 (d = 3, ..., 8)

Following the site's own convention (`docs/catalog/hypersimplex.md`): Δ(k,n) is a two-parameter family, and fixing k=2 (the smallest choice that isn't just the simplex) gives a one-parameter sequence indexed by dimension d = n-1. For each instance, this notebook:

1. lists all vertices (reduced to the full-dimensional chart — see `reduce_codim1`),
2. computes the canonical form via the general nbc method (Brown–Dupont Prop. 6.7), checking the defining pole-structure property,
3. computes the projective (polar) dual,
4. checks the **volume conjecture** at the centroid (n = 4–7 only — see below), see `simplex_explorer.ipynb`'s n=1 section for why it has to be the centroid,
5. enumerates all triangulations and identifies which are regular (n = 4, 5 only — see below),
6. computes the secondary polytope and its vertex embedding (n = 4, 5 only).

**Two separate computational ceilings here, not one.** Triangulation enumeration stops at n=5: n=6 (15 points) was measured to exceed 45 seconds, vs. 0.35s at n=5 (the hypersimplex is simplicial, not simple, so many nbc candidates compete per vertex). The volume conjecture stops at n=7, for an unrelated reason: it requires a *second* canonical-form computation in re-centered coordinates, which is measurably more expensive than the original chart (Δ(2,7): 1.7s original vs. 10.9s re-centered) and was extrapolated to take a minute or more by n=8/9. Plain canonical-form computation (no re-centering) stays cheap throughout — under 10s even at n=9 — so n=8 and n=9 still get the canonical form, pole-structure check, and dual, just not the volume conjecture.

**Requires the `sagemath` Jupyter kernel** and must be opened from the same synced folder as the `.sage` files — see `README.md`.

In [ ]:
load("general_canonical_forms.sage")

That `load` pulls in `common.sage` (vertex generators, `reduce_codim1`, `polar_dual`, `secondary_polytope_data`) and `vertex_sum_canonical_forms.sage` too, and runs `general_canonical_forms.sage`'s own test suite as a side effect (scroll up for that PASS/FAIL output; it already checks \u0394(2,4) and \u0394(2,5) via pole structure). Everything below is fresh, per-instance exploration of the hypersimplex family specifically.

## \u0394(2,4) — the octahedron (d = 3)

In [ ]:
n = 4
d = n - 1
y = [var(f"y{i}") for i in range(1, d + 1)]
pts = reduce_codim1(hypersimplex_vertices(2, n))
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Hypersimplex Delta(2,{n})", phi, P, y)
phi

### Canonical form, broken down by vertex

The single expression above is a sum over every vertex of P, and (the hypersimplex being simplicial, not simple) potentially several nbc terms *per* vertex, all combined into one expression — hard to read fast, and it hides the actual combinatorics the method is built on. `canonical_form_by_vertex` keeps every vertex's contribution separate instead: for each vertex, which facets (by index) meet there, how many (the **valency** — exactly d for a simple vertex, more otherwise), and each surviving nbc term's own small, factored contribution. Summing everything below reproduces the single expression above exactly.

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (taken at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
for t, gkz, is_reg in rows:
    print(t, "GKZ vector:", gkz, " regular:", is_reg)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()

## \u0394(2,5) — (d = 4)

In [ ]:
n = 5
d = n - 1
y = [var(f"y{i}") for i in range(1, d + 1)]
pts = reduce_codim1(hypersimplex_vertices(2, n))
P = Polyhedron(vertices=pts)
print(f"{len(pts)} vertices:")
pts

### Canonical form (Proposition 6.7, general nbc method)

In [ ]:
phi = general_canonical_form_density(P, y)
verify_pole_structure(f"Hypersimplex Delta(2,{n})", phi, P, y)
phi

### Canonical form, broken down by vertex

In [ ]:
rows = canonical_form_by_vertex(P, y)
print_canonical_form_by_vertex(rows)

### Projective dual

In [ ]:
Dual = polar_dual(P)
print(f"dual: {Dual.n_vertices()} vertices, {Dual.n_facets()} facets")
Dual.vertices_list()

### Volume conjecture: canonical form vs. the volume of the projective dual (taken at the centroid)

In [ ]:
centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
P_centered = Polyhedron(vertices=pts_centered)

phi_centroid = general_canonical_form_density(P_centered, y)
val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})

vol_dual = Dual.volume()
target = factorial(d) * vol_dual
print("phi at the centroid =", val_at_centroid)
print("d! * Vol(dual) =", target)
match_plus = bool((val_at_centroid - target) == 0)
match_minus = bool((val_at_centroid + target) == 0)
print("matches d! * Vol(dual):", match_plus, " matches -d! * Vol(dual):", match_minus)
assert match_plus or match_minus, "volume-conjecture identity failed -- would be a real bug"

### All triangulations, and which are regular

In [ ]:
sp, sp_reduced, rows = secondary_polytope_data(pts)
print(f"{len(rows)} triangulation(s) total, {sum(1 for _, _, r in rows if r)} regular")
rows[:5]

### Secondary polytope: vertex embedding

In [ ]:
print(f"secondary polytope: dimension {sp_reduced.dimension()}, {sp_reduced.n_vertices()} vertex/vertices")
sp_reduced.vertices_list()[:5]

## Δ(2,6), Δ(2,7) — (d = 5, 6)

Canonical form, dual, and the volume conjecture, for each — triangulation enumeration measured infeasible past n=5 (see the intro). Looped rather than repeated cell-by-cell, since there's nothing new to say about each beyond the numbers.

In [ ]:
for n in [6, 7]:
    d = n - 1
    y = [var(f"y{i}") for i in range(1, d + 1)]
    pts = reduce_codim1(hypersimplex_vertices(2, n))
    P = Polyhedron(vertices=pts)

    phi = general_canonical_form_density(P, y)
    pole_ok = verify_pole_structure(f"Hypersimplex Delta(2,{n})", phi, P, y)

    Dual = polar_dual(P)

    centroid = [sum(QQ(v[i]) for v in pts) / len(pts) for i in range(d)]
    pts_centered = [tuple(QQ(v[i]) - centroid[i] for i in range(d)) for v in pts]
    P_centered = Polyhedron(vertices=pts_centered)
    phi_centroid = general_canonical_form_density(P_centered, y)
    val_at_centroid = phi_centroid.subs({yi: 0 for yi in y})
    target = factorial(d) * Dual.volume()
    vol_ok = bool((val_at_centroid - target) == 0) or bool((val_at_centroid + target) == 0)

    print(f"n={n} (d={d}): {len(pts)} vertices, dual has {Dual.n_vertices()} vertices, "
          f"pole structure {'PASS' if pole_ok else 'FAIL'}, volume conjecture {'PASS' if vol_ok else 'FAIL'}")
    assert pole_ok and vol_ok, f"n={n}: a real check failed"

## Δ(2,8), Δ(2,9) — (d = 7, 8)

Canonical form and dual only here — **not** the volume conjecture. That check needs a *second* canonical-form computation, in re-centered coordinates, and re-centering was measured to make the computation substantially more expensive than in the original chart (Δ(2,7) alone: 1.7s original vs. 10.9s re-centered, while building this notebook) — extrapolating, Δ(2,8)/Δ(2,9) would likely each take a minute or more just for that second computation. The canonical form and pole-structure check (which don't need re-centering) stay cheap throughout, so those are shown for completeness.

In [ ]:
for n in [8, 9]:
    d = n - 1
    y = [var(f"y{i}") for i in range(1, d + 1)]
    pts = reduce_codim1(hypersimplex_vertices(2, n))
    P = Polyhedron(vertices=pts)

    phi = general_canonical_form_density(P, y)
    pole_ok = verify_pole_structure(f"Hypersimplex Delta(2,{n})", phi, P, y)
    Dual = polar_dual(P)

    print(f"n={n} (d={d}): {len(pts)} vertices, dual has {Dual.n_vertices()} vertices, "
          f"pole structure {'PASS' if pole_ok else 'FAIL'}")
    assert pole_ok, f"n={n}: pole-structure check failed"